Define the Forecasting Problem

Setting	- Value

Forecast Target	- Sales

Static Features	- StoreType, Assortment

Observed Past Features	- Sales, Open, Promo, SchoolHoliday

Known Future Features	- Promo, StateHoliday, SchoolHoliday, Month, DayOfWeek

Time Index	- Date

Entity ID	- Store ID

Historic Lookback Window	- 60 days (adjustable)

Forecast Prediction Window	30 days (adjustable)

These numbers (60/30 days) are standard starting points for retail.

For Store 111:

- Past 60 days' data (input sequence)
- Predict next 30 days' sales (output sequence)

Each Store becomes a mini time series.

Static Features are repeated along the sequence.

Observed Features (e.g., Sales, Promo history) are included.

Known Future Features (like planned Promo, Holidays) are provided.

3. Feature Categories for TFT

Type	Features

Static - Categorical Features	StoreType, Assortment

Static Numerical Features	- CompetitionDistance

Past Observed Features	- Sales, Open, Promo, SchoolHoliday

Known Future Feature - Promo, StateHoliday, SchoolHoliday, Month, DayOfWeek

Time-varying Features	- Date, WeekOfYear

This structure is very important for TFT to learn correctly.

##Starter Code to Prepare TFT Input Data

Step 1: Load Data and filter open days 

In [0]:
# Load the train master table
train = spark.table('rcg_store_catalog.rcg_store_schema.rossmann_store_sales_master_train').toPandas()
train = train[train['Open'] == 1]

Step 3: Sort and Create Time Index

In [0]:
# Make sure data is sorted by Store and Date
train = train.sort_values(['Store', 'Date'])

Step 4: Create Future Calendar
(When you predict, you need future dates + promo calendar etc.)

In [0]:
# Optional: Generate future dates for known future inputs
import pandas as pd

# Example: Create next 30 days for each store
future_dates = pd.date_range(train['Date'].max() + pd.Timedelta(days=1), periods=30)


Step 5: Define Feature Groups

In [0]:
# Feature Groups
static_categoricals = ['StoreType', 'Assortment']
static_numericals = ['CompetitionDistance']

observed_features = ['Sales', 'Promo', 'SchoolHoliday']
known_future_features = ['Promo', 'StateHoliday', 'SchoolHoliday', 'Month', 'DayOfWeek']


Step 6: Build Sequences
You can use libraries like PyTorch Forecasting or build sequences manually.

Here’s manual pseudo-logic:

In [0]:
sequence_data = []

# For each store
for store_id, store_data in train.groupby('Store'):
    
    store_data = store_data.sort_values('Date')
    
    for i in range(60, len(store_data) - 30):
        
        past_data = store_data.iloc[i-60:i]  # Past 60 days
        future_data = store_data.iloc[i:i+30]  # Next 30 days
        
        sequence_data.append({
            'store_id': store_id,
            'encoder_sales': past_data['Sales'].values,
            'encoder_promo': past_data['Promo'].values,
            'decoder_promo': future_data['Promo'].values,
            'decoder_holiday': future_data['StateHoliday'].values,
            'decoder_schoolholiday': future_data['SchoolHoliday'].values,
            'target_sales': future_data['Sales'].values,  # If training with known future sales
            'static_features': store_data[static_categoricals + static_numericals].iloc[0].to_dict()
        })


 Each sample = 1 training record for TFT
 Contains:

Past 60 days (encoder inputs)

Future 30 days (decoder known inputs)

Static store info

Target future sales

##Lets run a production ready code

In [0]:
# Install both packages
%pip install torch==1.13.1 pytorch-lightning==1.8.6 pytorch-forecasting==0.10.2 protobuf==3.20.3



In [0]:
dbutils.library.restartPython()

In [0]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from pytorch_lightning import Trainer
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

# 1. Load Train Data
train = spark.table('rcg_store_catalog.rcg_store_schema.rossmann_store_sales_master_train').toPandas()

# 2. Preprocessing
train = train[train['Open'] == 1]  # Only Open stores
train['Date'] = pd.to_datetime(train['Date'])
train = train.sort_values(['Store', 'Date'])
train['time_idx'] = (train['Date'] - train['Date'].min()).dt.days

# Feature Engineering
train['month'] = train['Date'].dt.month
train['day_of_week'] = train['Date'].dt.dayofweek
train['Sales_log'] = np.log1p(train['Sales'])  # log1p(Sales)

# Fix categorical features
train['Promo2'] = train['Promo2'].astype(str)
train['StateHoliday'] = train['StateHoliday'].astype(str)
train['CompetitionDistance'] = train['CompetitionDistance'].fillna(20000)

# 3. Define TimeSeriesDataSet
max_encoder_length = 60  # past days
max_prediction_length = 30  # future days

sampled_stores = train['Store'].drop_duplicates().sample(100, random_state=42)
train_small = train[train['Store'].isin(sampled_stores)]

training = TimeSeriesDataSet(
    train_small,
    time_idx="time_idx",
    target="Sales_log",
    group_ids=["Store"],
    static_categoricals=["StoreType", "Assortment"],
    time_varying_known_categoricals=["StateHoliday", "Promo2"],
    time_varying_known_reals=["time_idx", "month", "day_of_week", "CompetitionDistance"],
    time_varying_unknown_reals=["Sales_log", "Promo", "SchoolHoliday"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    target_normalizer=GroupNormalizer(groups=["Store"]),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True
)

# 4. Create DataLoader
from torch.utils.data import DataLoader
batch_size = 32
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)

# 5. Build the TFT Model
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.03,
    hidden_size=8,
    attention_head_size=1,
    dropout=0.1,
    hidden_continuous_size=8,
    output_size=7,
    loss=QuantileLoss(),
    log_interval=10,
    reduce_on_plateau_patience=4,
)

# 6. Train the Model
trainer = Trainer(
    max_epochs=2,
    accelerator="cpu",      # Use "cpu" unless you have GPU cluster
    gradient_clip_val=0.1,
)

trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
)

# 7. Predict on Training Data
actuals = torch.cat([y[0] for x, y in iter(train_dataloader)])
predictions = tft.predict(train_dataloader)

# 8. Plot Actuals vs Predictions
plt.figure(figsize=(10,6))
plt.scatter(np.expm1(actuals.numpy()), np.expm1(predictions.numpy()), alpha=0.5)
plt.plot([0, np.expm1(actuals).max()], [0, np.expm1(actuals).max()], color='red', linestyle='--')
plt.xlabel('Actual Sales')
plt.ylabel('Predicted Sales')
plt.title('Actual vs Predicted Sales (Validation Set)')
plt.grid(True)
plt.show()


Takes trained model (tft) 

Takes past train DataFrame (train) 

Prepares next 30 days future dataset automatically 

Returns predicted sales for next 30 days 



In [0]:
def forecast_next_30_days(tft_model, train_df, max_encoder_length=60, max_prediction_length=30, batch_size=64):
    """
    Forecast next 30 days sales using trained Temporal Fusion Transformer model.
    
    Args:
        tft_model: trained TemporalFusionTransformer model
        train_df: Pandas DataFrame with historical data (used for training)
        max_encoder_length: number of past days model expects
        max_prediction_length: number of future days to predict
        batch_size: batch size for prediction dataloader
    
    Returns:
        Tuple of (predicted_sales_array, future_full_dataframe)
    """
    # Step 1: Prepare future dates
    max_training_date = train_df['Date'].max()
    future_dates = pd.date_range(start=max_training_date + pd.Timedelta(days=1), periods=max_prediction_length)
    stores = train_df['Store'].unique()
    
    future = pd.DataFrame([(store, date) for store in stores for date in future_dates], columns=['Store', 'Date'])
    future['time_idx'] = (future['Date'] - train_df['Date'].min()).dt.days
    future['month'] = future['Date'].dt.month
    future['day_of_week'] = future['Date'].dt.dayofweek
    future['Promo'] = 0  # Assume no promotions
    future['Promo2'] = '0'
    future['StateHoliday'] = '0'
    future['SchoolHoliday'] = 0

    # Merge static store features
    static_features = train_df.drop_duplicates('Store')[['Store', 'StoreType', 'Assortment', 'CompetitionDistance']]
    future = future.merge(static_features, on='Store', how='left')
    future['CompetitionDistance'] = future['CompetitionDistance'].fillna(20000)
    
    # Add dummy Sales_log for future (needed by TimeSeriesDataSet)
    future['Sales_log'] = 0.0

    # Step 2: Get last 60 days history for each Store
    history = (
        train_df.sort_values(["Store", "Date"])
        .groupby("Store")
        .tail(max_encoder_length)
        .reset_index(drop=True)
    )

    # Step 3: Combine history and future
    full_future = pd.concat([history, future], axis=0)

    # ✅ Important: Reset index cleanly!
    full_future = full_future.reset_index(drop=True)

    # Step 4: Create prediction TimeSeriesDataSet
    prediction_dataset = TimeSeriesDataSet(
        full_future,
        time_idx="time_idx",
        target="Sales_log",
        group_ids=["Store"],
        static_categoricals=["StoreType", "Assortment"],
        time_varying_known_categoricals=["StateHoliday", "Promo2"],
        time_varying_known_reals=["time_idx", "month", "day_of_week", "CompetitionDistance"],
        time_varying_unknown_reals=["Sales_log", "Promo", "SchoolHoliday"],
        max_encoder_length=max_encoder_length,
        max_prediction_length=max_prediction_length,
        target_normalizer=GroupNormalizer(groups=["Store"]),
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
        allow_missing_timesteps=True
    )

    # Step 5: Create DataLoader
    prediction_dataloader = prediction_dataset.to_dataloader(train=False, batch_size=batch_size, num_workers=0)

    # Step 6: Predict
    raw_predictions, x = tft_model.predict(prediction_dataloader, mode="raw", return_x=True)

    # Step 7: Extract P50 predictions
    predicted_sales_log = raw_predictions["prediction"][:, :, 2]  # 2 = 50% quantile (median)
    predicted_sales = np.expm1(predicted_sales_log.cpu().numpy())  # Undo log1p

    return predicted_sales, future


In [0]:
# Call forecast method
predicted_sales, future_calendar = forecast_next_30_days(tft, train)

# Check shape
print(predicted_sales.shape)  # (number of stores, 30)

# Example: Plot forecast for a random store
import matplotlib.pyplot as plt

store_idx = 0  # Pick a store index

plt.figure(figsize=(12,6))
plt.plot(predicted_sales[store_idx], marker='o')
plt.title(f"Forecasted Sales for Store {future_calendar['Store'].unique()[store_idx]} (Next 30 Days)")
plt.xlabel('Day')
plt.ylabel('Predicted Sales')
plt.grid()
plt.show()


## Next 30 Days with promo

In [0]:
def forecast_next_30_days_with_promo(tft_model, train_df, max_encoder_length=60, max_prediction_length=30, batch_size=64):
    """
    Forecast next 30 days sales with Promo ON every Saturday using trained Temporal Fusion Transformer model.
    
    Args:
        tft_model: trained TemporalFusionTransformer model
        train_df: Pandas DataFrame with historical data (used for training)
        max_encoder_length: number of past days model expects
        max_prediction_length: number of future days to predict
        batch_size: batch size for prediction dataloader
    
    Returns:
        Tuple of (predicted_sales_array, future_full_dataframe_with_promo)
    """
    # Step 1: Prepare future dates
    max_training_date = train_df['Date'].max()
    future_dates = pd.date_range(start=max_training_date + pd.Timedelta(days=1), periods=max_prediction_length)
    stores = train_df['Store'].unique()
    
    future = pd.DataFrame([(store, date) for store in stores for date in future_dates], columns=['Store', 'Date'])
    future['time_idx'] = (future['Date'] - train_df['Date'].min()).dt.days
    future['month'] = future['Date'].dt.month
    future['day_of_week'] = future['Date'].dt.dayofweek

    # Step 2: Set known future inputs
    future['Promo'] = 0  # default no promo
    future['Promo2'] = '0'
    future['StateHoliday'] = '0'
    future['SchoolHoliday'] = 0

    # ✅ Set Promo = 1 for Saturdays (day_of_week == 5)
    future.loc[future['day_of_week'] == 5, 'Promo'] = 1

    # Update Promo2 also if needed (make it '1' as string if Promo=1)
    future['Promo2'] = future['Promo'].astype(str)

    # Step 3: Merge static store features
    static_features = train_df.drop_duplicates('Store')[['Store', 'StoreType', 'Assortment', 'CompetitionDistance']]
    future = future.merge(static_features, on='Store', how='left')
    future['CompetitionDistance'] = future['CompetitionDistance'].fillna(20000)
    
    # Add dummy Sales_log for future (needed by TimeSeriesDataSet)
    future['Sales_log'] = 0.0

    # Step 4: Get last 60 days history
    history = (
        train_df.sort_values(["Store", "Date"])
        .groupby("Store")
        .tail(max_encoder_length)
        .reset_index(drop=True)
    )

    # Step 5: Combine history + future
    full_future = pd.concat([history, future], axis=0)
    full_future = full_future.reset_index(drop=True)  # ✅ Important to reset index!

    # Step 6: Create prediction TimeSeriesDataSet
    prediction_dataset = TimeSeriesDataSet(
        full_future,
        time_idx="time_idx",
        target="Sales_log",
        group_ids=["Store"],
        static_categoricals=["StoreType", "Assortment"],
        time_varying_known_categoricals=["StateHoliday", "Promo2"],
        time_varying_known_reals=["time_idx", "month", "day_of_week", "CompetitionDistance"],
        time_varying_unknown_reals=["Sales_log", "Promo", "SchoolHoliday"],
        max_encoder_length=max_encoder_length,
        max_prediction_length=max_prediction_length,
        target_normalizer=GroupNormalizer(groups=["Store"]),
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
        allow_missing_timesteps=True
    )

    # Step 7: Create DataLoader
    prediction_dataloader = prediction_dataset.to_dataloader(train=False, batch_size=batch_size, num_workers=0)

    # Step 8: Predict
    raw_predictions, x = tft_model.predict(prediction_dataloader, mode="raw", return_x=True)

    # Step 9: Extract P50 predictions
    predicted_sales_log = raw_predictions["prediction"][:, :, 2]  # 2 = 50% quantile (median)
    predicted_sales = np.expm1(predicted_sales_log.cpu().numpy())  # Undo log1p

    return predicted_sales, future


In [0]:
# Call Promo scenario forecast
promo_forecast_sales, promo_future_calendar = forecast_next_30_days_with_promo(tft, train)

# Check shape
print(promo_forecast_sales.shape)  # (number of stores, 30)

# Example: Plot forecast for a store
import matplotlib.pyplot as plt

store_idx = 0

plt.figure(figsize=(12,6))
plt.plot(promo_forecast_sales[store_idx], marker='o')
plt.title(f"Promo-ON Forecast for Store {promo_future_calendar['Store'].unique()[store_idx]} (Next 30 Days)")
plt.xlabel('Day')
plt.ylabel('Predicted Sales with Promo Saturdays')
plt.grid()
plt.show()


 School Holidays What-If Forecasting

  Simulate impact if there are School Holidays during the next 30 days
  (e.g., Assume all Fridays are School Holidays.)

  This is very useful for retail:

  School Holidays → families at home → shopping changes!

In [0]:
def forecast_next_30_days_with_school_holidays(tft_model, train_df, max_encoder_length=60, max_prediction_length=30, batch_size=64):
    """
    Forecast next 30 days sales with School Holidays every Friday using trained Temporal Fusion Transformer model.
    
    Args:
        tft_model: trained TemporalFusionTransformer model
        train_df: Pandas DataFrame with historical data (used for training)
        max_encoder_length: number of past days model expects
        max_prediction_length: number of future days to predict
        batch_size: batch size for prediction dataloader
    
    Returns:
        Tuple of (predicted_sales_array, future_full_dataframe_with_school_holiday)
    """
    # Step 1: Prepare future dates
    max_training_date = train_df['Date'].max()
    future_dates = pd.date_range(start=max_training_date + pd.Timedelta(days=1), periods=max_prediction_length)
    stores = train_df['Store'].unique()
    
    future = pd.DataFrame([(store, date) for store in stores for date in future_dates], columns=['Store', 'Date'])
    future['time_idx'] = (future['Date'] - train_df['Date'].min()).dt.days
    future['month'] = future['Date'].dt.month
    future['day_of_week'] = future['Date'].dt.dayofweek

    # Step 2: Set known future inputs
    future['Promo'] = 0
    future['Promo2'] = '0'
    future['StateHoliday'] = '0'
    future['SchoolHoliday'] = 0  # default

    # ✅ Set SchoolHoliday = 1 for Fridays (day_of_week == 4)
    future.loc[future['day_of_week'] == 4, 'SchoolHoliday'] = 1

    # Step 3: Merge static store features
    static_features = train_df.drop_duplicates('Store')[['Store', 'StoreType', 'Assortment', 'CompetitionDistance']]
    future = future.merge(static_features, on='Store', how='left')
    future['CompetitionDistance'] = future['CompetitionDistance'].fillna(20000)
    
    # Add dummy Sales_log for future (needed by TimeSeriesDataSet)
    future['Sales_log'] = 0.0

    # Step 4: Get last 60 days history
    history = (
        train_df.sort_values(["Store", "Date"])
        .groupby("Store")
        .tail(max_encoder_length)
        .reset_index(drop=True)
    )

    # Step 5: Combine history + future
    full_future = pd.concat([history, future], axis=0)
    full_future = full_future.reset_index(drop=True)  # ✅ Important!

    # Step 6: Create prediction TimeSeriesDataSet
    prediction_dataset = TimeSeriesDataSet(
        full_future,
        time_idx="time_idx",
        target="Sales_log",
        group_ids=["Store"],
        static_categoricals=["StoreType", "Assortment"],
        time_varying_known_categoricals=["StateHoliday", "Promo2"],
        time_varying_known_reals=["time_idx", "month", "day_of_week", "CompetitionDistance"],
        time_varying_unknown_reals=["Sales_log", "Promo", "SchoolHoliday"],
        max_encoder_length=max_encoder_length,
        max_prediction_length=max_prediction_length,
        target_normalizer=GroupNormalizer(groups=["Store"]),
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
        allow_missing_timesteps=True
    )

    # Step 7: Create DataLoader
    prediction_dataloader = prediction_dataset.to_dataloader(train=False, batch_size=batch_size, num_workers=0)

    # Step 8: Predict
    raw_predictions, x = tft_model.predict(prediction_dataloader, mode="raw", return_x=True)

    # Step 9: Extract P50 predictions
    predicted_sales_log = raw_predictions["prediction"][:, :, 2]
    predicted_sales = np.expm1(predicted_sales_log.cpu().numpy())

    return predicted_sales, future


In [0]:
# Call School Holiday scenario forecast
school_holiday_forecast_sales, school_holiday_future_calendar = forecast_next_30_days_with_school_holidays(tft, train)

# Check shape
print(school_holiday_forecast_sales.shape)  # (number of stores, 30)

# Example: Plot forecast for a store
import matplotlib.pyplot as plt

store_idx = 0

plt.figure(figsize=(12,6))
plt.plot(school_holiday_forecast_sales[store_idx], marker='o')
plt.title(f"School Holiday-ON Forecast for Store {school_holiday_future_calendar['Store'].unique()[store_idx]} (Next 30 Days)")
plt.xlabel('Day')
plt.ylabel('Predicted Sales with School Holidays on Fridays')
plt.grid()
plt.show()


Promo running on all weekend

In [0]:
  def forecast_next_30_days_promo_weekend(tft_model, train_df, max_encoder_length=60, max_prediction_length=30, batch_size=64):
    """
    Forecast next 30 days sales with Promo ON every Saturday and Sunday using trained Temporal Fusion Transformer model.
    
    Args:
        tft_model: trained TemporalFusionTransformer model
        train_df: Pandas DataFrame with historical data (used for training)
        max_encoder_length: number of past days model expects
        max_prediction_length: number of future days to predict
        batch_size: batch size for prediction dataloader
    
    Returns:
        Tuple of (predicted_sales_array, future_full_dataframe_with_weekend_promo)
    """
    # Step 1: Prepare future dates
    max_training_date = train_df['Date'].max()
    future_dates = pd.date_range(start=max_training_date + pd.Timedelta(days=1), periods=max_prediction_length)
    stores = train_df['Store'].unique()
    
    future = pd.DataFrame([(store, date) for store in stores for date in future_dates], columns=['Store', 'Date'])
    future['time_idx'] = (future['Date'] - train_df['Date'].min()).dt.days
    future['month'] = future['Date'].dt.month
    future['day_of_week'] = future['Date'].dt.dayofweek + 1  # Rossmann format: 1=Monday, 7=Sunday

    # Step 2: Set known future inputs
    future['Promo'] = 0
    future['Promo2'] = '0'
    future['StateHoliday'] = '0'
    future['SchoolHoliday'] = 0

    # ✅ Set Promo = 1 for Saturdays (day_of_week==6) and Sundays (day_of_week==7)
    future.loc[future['day_of_week'].isin([6, 7]), 'Promo'] = 1
    future['Promo2'] = future['Promo'].astype(str)

    # Step 3: Merge static store features
    static_features = train_df.drop_duplicates('Store')[['Store', 'StoreType', 'Assortment', 'CompetitionDistance']]
    future = future.merge(static_features, on='Store', how='left')
    future['CompetitionDistance'] = future['CompetitionDistance'].fillna(20000)
    
    # Add dummy Sales_log for future
    future['Sales_log'] = 0.0

    # Step 4: Get last 60 days history
    history = (
        train_df.sort_values(["Store", "Date"])
        .groupby("Store")
        .tail(max_encoder_length)
        .reset_index(drop=True)
    )

    # Step 5: Combine history + future
    full_future = pd.concat([history, future], axis=0)
    full_future = full_future.reset_index(drop=True)

    # Step 6: Create prediction TimeSeriesDataSet
    prediction_dataset = TimeSeriesDataSet(
        full_future,
        time_idx="time_idx",
        target="Sales_log",
        group_ids=["Store"],
        static_categoricals=["StoreType", "Assortment"],
        time_varying_known_categoricals=["StateHoliday", "Promo2"],
        time_varying_known_reals=["time_idx", "month", "day_of_week", "CompetitionDistance"],
        time_varying_unknown_reals=["Sales_log", "Promo", "SchoolHoliday"],
        max_encoder_length=max_encoder_length,
        max_prediction_length=max_prediction_length,
        target_normalizer=GroupNormalizer(groups=["Store"]),
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
        allow_missing_timesteps=True
    )

    # Step 7: Create DataLoader
    prediction_dataloader = prediction_dataset.to_dataloader(train=False, batch_size=batch_size, num_workers=0)

    # Step 8: Predict
    raw_predictions, x = tft_model.predict(prediction_dataloader, mode="raw", return_x=True)

    # Step 9: Extract P50 predictions
    predicted_sales_log = raw_predictions["prediction"][:, :, 2]
    predicted_sales = np.expm1(predicted_sales_log.cpu().numpy())

    return predicted_sales, future


In [0]:
# Call Promo Every Weekend forecast
promo_weekend_forecast_sales, promo_weekend_future_calendar = forecast_next_30_days_promo_weekend(tft, train)

# Check
print(promo_weekend_forecast_sales.shape)  # (number of stores, 30)

# Example: Plot forecast for a store
import matplotlib.pyplot as plt

store_idx = 0

plt.figure(figsize=(12,6))
plt.plot(promo_weekend_forecast_sales[store_idx], marker='o')
plt.title(f"Promo Every Weekend Forecast for Store {promo_weekend_future_calendar['Store'].unique()[store_idx]} (Next 30 Days)")
plt.xlabel('Day')
plt.ylabel('Predicted Sales (Promo on Sat-Sun)')
plt.grid()
plt.show()
